# Phase 1 - Week 1-2: PaySim 데이터 탐색

## 문제 정의

모바일 머니 거래 데이터 600만 건에서 **사기 거래를 어떻게 식별할 수 있는가**를 탐구한다.

**왜 이 문제가 중요한가**
- 모바일 결제 시장 급성장에 따라 사기 거래 규모도 증가
- 사기 한 건당 평균 손실액이 크고, 사후 적발은 회수가 어려움
- 사전 탐지 모델은 핀테크 회사의 핵심 리스크 관리 수단

**이번 노트북의 목표**
1. 데이터 구조와 컬럼의 의미 파악
2. 사기 거래의 비율과 분포 확인 (불균형 정도 체감)
3. 사기 거래와 정상 거래의 차이를 만드는 변수 후보 탐색

**다음 노트북으로 이어질 질문**
- 어떤 변수가 사기 거래를 가장 잘 구분하는가?
- 시간대별/거래유형별 사기 패턴이 있는가?
- 단순 규칙(rule-based)으로 어디까지 잡을 수 있고, ML이 필요한 지점은 어디인가?


In [3]:
import pandas as pd
from pathlib import Path

# 프로젝트 루트 기준 데이터 경로
DATA_PATH = Path("../data/PS_20174392719_1491204439457_log.csv")
DATA_PATH.exists()

True

In [4]:
%%time
df = pd.read_csv(DATA_PATH)
df.shape

CPU times: total: 11.7 s
Wall time: 12 s


(6362620, 11)

In [5]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [7]:
df.tail()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0
6362619,743,CASH_OUT,850002.52,C1280323807,850002.52,0.0,C873221189,6510099.11,7360101.63,1,0


In [8]:
df["isFraud"].value_counts(normalize=False)

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [9]:
df["isFraud"].value_counts(normalize=True)

isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

## 핵심 관찰: 극심한 불균형

전체 600만 건 중 사기는 8,213건 (**0.13%**). 정상:사기 = 약 770:1.

**시사점**
- "사기 아님"으로만 예측해도 정확도 **99.87%** - 정확도(accuracy)는 무의미한 지표
- 평가는 **Precision / Recall / F1 / AUC**로 해야 함
- 학습 시 **불균형 처리 필요** (SMOTE, undersampling, class_weight 등)

**비즈니스 트레이드오프**
- Recall ↑ (사기를 많이 잡기) → False Positive 증가 → 정상 고객 불편
- Precision ↑ (잡은 게 진짜 사기) -> False Negative 증가 → 사기 놓침
- 핀테크에서는 보통 **Recall을 우선시**하되 운영 가능한 Precision 수준 유지가 목표

In [10]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [11]:
df.groupby("type")["isFraud"].sum()

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64

In [12]:
df.groupby("type")["isFraud"].mean().sort_values(ascending=False)

type
TRANSFER    0.007688
CASH_OUT    0.001840
CASH_IN     0.000000
DEBIT       0.000000
PAYMENT     0.000000
Name: isFraud, dtype: float64

## 핵심 발견: 사기는 2가지 거래 유형에만 존재

| 거래 유형 | 전체 건수 | 사기 건수 | 사기율 |
|---|---|---|---|
| TRANSFER | 532,909 | 4,097 | 0.77% |
| CASH_OUT | 2,237,500 | 4,116 | 0.18% |
| CASH_IN | 1,399,284 | 0 | 0% |
| DEBIT | 41,432 | 0 | 0% |
| PAYMENT | 2,151,495 | 0 | 0% |

**시사점**
- 전체 8,213건의 사기가 **TRANSFER + CASH_OUT에 100% 집중**
- 다른 3개 유형(CASH_IN, DEBIT, PAYMENT)은 분석/모델링 대상에서 제외 가능
- 실제 분석 대상은 약 277만 건으로 축소 (전체의 43%)

**가설: 사기의 본질적 패턴**
`df.head()`에서 본 index 2-3행:
-TRANSFER 181 (사기) -> 같은 금액 CASH_OUT 181 (사기)
- 즉 **"송금받자마자 현금화"**가 핵심 사기 시나리오로 추정됨
- 다음 분석에서 검증: 동일 금액의 TRANSFER → CASH_OUT 쌍이 사기와 강하게 연관되는가?

In [14]:
fraud_df = df[df["isFraud"] == 1].copy()
fraud_df.shape

(8213, 11)

In [15]:
fraud_df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.00,C1305486145,181.00,0.0,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.0,C38997010,21182.0,0.00,1,0
251,1,TRANSFER,2806.00,C1420196421,2806.00,0.0,C972765878,0.0,0.00,1,0
252,1,CASH_OUT,2806.00,C2101527076,2806.00,0.0,C1007251739,26202.0,0.00,1,0
680,1,TRANSFER,20128.00,C137533655,20128.00,0.0,C1848415041,0.0,0.00,1,0
681,1,CASH_OUT,20128.00,C1118430673,20128.00,0.0,C339924917,6268.0,12145.85,1,0
724,1,CASH_OUT,416001.33,C749981943,0.00,0.0,C667346055,102.0,9291619.62,1,0
969,1,TRANSFER,1277212.77,C1334405552,1277212.77,0.0,C431687661,0.0,0.00,1,0
970,1,CASH_OUT,1277212.77,C467632528,1277212.77,0.0,C716083600,0.0,2444985.19,1,0
1115,1,TRANSFER,35063.63,C1364127192,35063.63,0.0,C1136419747,0.0,0.00,1,0


In [16]:
fraud_df["amount"].describe()

count    8.213000e+03
mean     1.467967e+06
std      2.404253e+06
min      0.000000e+00
25%      1.270913e+05
50%      4.414234e+05
75%      1.517771e+06
max      1.000000e+07
Name: amount, dtype: float64

In [17]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


In [18]:
fraud_df["amount_ratio"] = fraud_df["amount"] / fraud_df["oldbalanceOrg"]
fraud_df["amount_ratio"].describe()

C:\dev\fintech-data-analysis\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count    8197.000000
mean             inf
std              NaN
min         0.038373
25%         1.000000
50%         1.000000
75%         1.000000
max              inf
Name: amount_ratio, dtype: float64

In [24]:
# TRANSFER + CASH_OUT만 (사기 가능한 유형만)
mask = df["type"].isin(["TRANSFER", "CASH_OUT"])
subset = df[mask].copy()

import numpy as np
subset["amount_ratio"] = subset["amount"] / subset["oldbalanceOrg"].replace(0, np.nan)
subset.groupby("isFraud")["amount_ratio"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,1453655.0,277.373667,6845.428578,0.000003,1.513163,5.775815,23.670412,3.925476e+06
1,8172.0,0.998765,0.385326,0.038373,1.000000,1.000000,1.000000,2.952662e+01


In [25]:
# 사기 TRANSFER와 사기 CASH_OUT의 금액 분포 매칭
fraud_transfer = fraud_df[fraud_df["type"] == "TRANSFER"]["amount"]
fraud_cashout = fraud_df[fraud_df["type"] == "CASH_OUT"]["amount"]

print(f"사기 TRANSFER 건수: {len(fraud_transfer)}")
print(f"사기 CASH_OUT 건수: {len(fraud_cashout)}")

# 두 금액 집합의 교집합 (같은 금액이 양쪽에 다 있는지)
common = set(fraud_transfer).intersection(set(fraud_cashout))
print(f"양쪽에 같은 금액으로 등장: {len(common)}")

사기 TRANSFER 건수: 4097
사기 CASH_OUT 건수: 4116
양쪽에 같은 금액으로 등장: 3932


## Phase 1 핵심 발견 - 사기 거래의 3대 시그니처

| # | 패턴 | 정량 결과 |
|---|---|---|
| 1 | 거래 유형이 TRANSFER 또는 CASH_OUT | 8,213건 중 **100%** |
| 2 | `amount/oldbalanceOrg = 1.0` (전액 송금) | 사기 75% 분위까지 **정확히 1.0**, 평균 0.998 | 
| 3 | TRANSFER ↔ 동일 금액 CASH_OUT 쌍 발생 | 4,097건 중 **3,932건 (96%)** |

### 정상 거래와의 비교 (amount_ratio)

| 분위 | 정상 | 사기 |
|---|---|---|
| 25% | 1.51 | **1.00** |
| 50% | 5.78 | **1.00** |
| 75% | 23.67 | **1.00** |
| 평균 | 277.37 | **0.998** |

→ 사기 거래는 amount_ratio가 1.0에 극도로 집중되어 있는 반면, 정상 거래는 광범위하게 분산.
→ **`amount_ratio` 변수 하나만으로도 강력한 분류 시그널.**

### 시사점

- 단순 규칙 (예: `type ∈ {TRANSFER, CASH_OUT}` AND `amount/oldbalanceOrg ≈ 1.0`)으로 높은 Recall 가능
- Week 5-6에서 이 규칙 기반 베이스라인 vs ML 모델 비교가 핵심 평가 포인트가 됨
- 새로 파생할 feature: `amount_ratio = amount / oldbalanceOrg`

### 데이터 한계 (재확인)

- 사기 거래 max 금액이 정확히 10,000,000으로 천장(cap) → PaySim 시뮬레이터의 인공성
- 정상 거래의 `oldbalanceOrg`에 매우 작은 값들이 많아 amount_ratio가 인위적으로 커지는 경향
- 실제 핀테크 환경에서는 더 노이즈가 많을 것 → 모델이 패턴 단순성에 과적합되지 않도록 주의 

In [26]:
df["step"].describe()

count    6.362620e+06
mean     2.433972e+02
std      1.423320e+02
min      1.000000e+00
25%      1.560000e+02
50%      2.390000e+02
75%      3.350000e+02
max      7.430000e+02
Name: step, dtype: float64

In [27]:
# step을 24로 나눈 나머지 = 하루 중 시간
df["hour"] = df["step"] % 24

# 시간대별 전체 거래 vs 사기 거래
hourly = df.groupby("hour").agg(
    total=("isFraud", "count"),
    fraud=("isFraud", "sum"),
)
hourly["fraud_rate"] = hourly["fraud"] / hourly["total"]
hourly

,total,fraud,fraud_rate
hour,,,
0,71587,300,0.004191
1,27111,358,0.013205
2,9018,372,0.041251
3,2007,326,0.162431
4,1241,274,0.220790
5,1641,366,0.223035
6,3420,358,0.104678
7,8988,328,0.036493
8,26915,368,0.013673


In [30]:
# step을 24로 나눈 몫 = 며칠째
df["day"] = df["step"] // 24
daily_fraud = df.groupby("day")["isFraud"].sum()
print(f"기간: {df['step'].min()}~{df['step'].max()} step ({(df['step'].max() // 24) + 1}일)")
print(f"일평균 사기 건수: {daily_fraud.mean():.1f}")
print(f"일별 사기 건수 std: {daily_fraud.std():.1f}")

기간: 1~743 step (31일)
일평균 사기 건수: 264.9
일별 사기 건수 std: 22.5


## 시간 패턴 발견: 새벽 거래의 위험성

### 시간대별 사기율 (Top/Bottom)

| 시간 | 전체 거래 | 사기 건수 | 사기율 |
|---|---|---|---|
| **새벽 5시** | 1,641 | 366 | **22.3%** |
| **새벽 4시** | 1,241 | 274 | **22.1%** |
| 새벽 3시 | 2,007 | 326 | 16.2% |
| 새벽 6시 | 3,420 | 358 | 10.5% |
| 정오 12시 | 483,418 | 339 | 0.07% |
| 저녁 19시 | 647,814 | 342 | **0.05%** |

### 핵심 통찰

- **사기 건수 자체는 시간대별 균일** (300-370건/시간)
- **정상 거래는 영업시간 (9-21시) 폭증**, 새벽엔 1/300 수준으로 급감
- 결과: 새벽 사기율이 저녁 사기율의 **약 400배**

### 비즈니스 해석

- 사기범은 24시간 균등 활동, 일반 사용자는 새벽 활동 없음
- **야간 거래(3-7시)는 정상이어도 위험 가중치 부여 합리적**
- 실무 핀테크 사기 탐지 시스템의 "야간 거래 룰"이 데이터로 검증됨

### 새로 추가할 feature

- `hour` = `step % 24` - 시간대별 위험도 학습용
- `is_night` = (3 ≤ hour ≤ 7) - 야간 플래그 (선택적)

### 일자별 패턴

- 31일 기간, 일평균 사기 264.9건 (std 22.5)
- 특정 요일/날짜 집중 없음 → 사기는 **일정한 시간 패턴**으로 발생

## Week 1-2 마무리: 발견 요약과 다음 단계

### 발견한 4가지 핵심 시그니처

1. **거래 유형**: 사기는 TRANSFER + CASH_OUT에 100% 집중
2. **잔액 패턴**: `amount/oldbalanceOrg ≈ 1.0` (전액 송금) - 사기 75% 정확히 1.0
3. **연속성**: TRANSFER와 동일 금액 CASH_OUT 쌍 (96% 매칭)
4. **시간대**: 새벽 4-5시 사기율 22%, 저녁 19시 사기율 0.05% (400배 차이)

### 다음 노트북(02_sql_analysis.ipynb)에서 할 일

1. SQLite에 데이터 적재 → SQL로 같은 분석 재현
2. SQL 윈도우 함수로 **고객별 거래 패턴 추적** (동일 nameOrig의 이전 거래 비교)
3. 단순 규칙 기반 분류기의 Precision/Recall 측정
   - Rule: `type in (TRANSFER, CASH_OUT) AND amount_ratio ≈ 1.0`
4. Week 5-6 ML 모델의 베이스라인으로 사용

### 다음 분석에서 검증할 질문

- 단순 규칙만으로 Recall 75%, Precision 어느 수준까지 가능한가?
- 시간(hour) feature를 추가하면 Precision이 얼마나 올라가는가?
- 동일 송금자가 짧은 시간 내 여러 건 거래 시 사기 가능성이 높은가?
- `oldbalanceDest`(수취자 잔액)에서 추가 시그널을 찾을 수 있는가?

### 데이터 한계 종합

- 합성 데이터의 인공성: 사기 금액 cap (10M), 단순한 패턴, 잔액 컬럼 노이즈
- 모델이 이 인공성에 과적합되지 않도록 주의
- 실제 핀테크 환경에서는 더 복잡한 노이즈와 진화하는 사기 패턴 존재